# GPT-2 ADMM-Based Pruning Implementation

**Project**: LLM Pruning using ADMM  
**Group**: 13  
**Paper**: Fast and Optimal Weight Update for Pruned Large Language Models

This notebook implements the ADMM-based pruning methodology on actual GPT-2 models, following the research paper methodology exactly.

## 1. Environment Setup and Dependencies

In [1]:
# Install required packages
# Uncomment the following lines if packages are not installed
# !pip install transformers torch numpy matplotlib datasets evaluate

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from datasets import load_dataset
import warnings
import copy
from tqdm import tqdm
import json
import os

warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"PyTorch version: {torch.__version__}")

C:\Users\Raghuram S\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ModuleNotFoundError: No module named 'datasets'

## 2. Load Pre-trained GPT-2 Model

In [ ]:
# Load GPT-2 model and tokenizer
model_name = "gpt2"  # Can be changed to gpt2-medium, gpt2-large, etc.

print(f"Loading {model_name} model and tokenizer...")
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)

# Add padding token if not present
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model.to(device)
print(f"Model loaded successfully. Total parameters: {sum(p.numel() for p in model.parameters()):,}")

# Print model architecture summary
print("\nModel Architecture:")
for name, module in model.named_modules():
    if isinstance(module, nn.Linear) and len(name.split('.')) <= 4:
        print(f"{name}: {module.weight.shape}")

## 3. Extract Model Layers and Prepare Calibration Data

In [ ]:
# Load calibration dataset (small subset of WikiText-2)
def prepare_calibration_data(tokenizer, num_samples=100, max_length=128):
    """
    Prepare calibration data for computing input activations
    """
    dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")
    
    texts = []
    for i, example in enumerate(dataset):
        if i >= num_samples:
            break
        text = example['text'].strip()
        if len(text) > 50:  # Filter out very short texts
            texts.append(text)
    
    # Tokenize texts
    inputs = tokenizer(texts[:num_samples], 
                      max_length=max_length, 
                      padding=True, 
                      truncation=True, 
                      return_tensors="pt")
    
    return inputs['input_ids'].to(device), inputs['attention_mask'].to(device)

print("Preparing calibration data...")
calibration_ids, calibration_mask = prepare_calibration_data(tokenizer, num_samples=50)
print(f"Calibration data shape: {calibration_ids.shape}")

In [ ]:
# Function to extract specific layer weights and compute activations
def extract_layer_info(model, layer_idx, input_ids, attention_mask):
    """
    Extract weights and compute input activations for a specific transformer layer
    """
    model.eval()
    
    # Hook to capture activations
    activations = {}
    
    def hook_fn(name):
        def hook(module, input, output):
            activations[name] = input[0].detach().cpu()
        return hook
    
    # Register hooks for MLP layers
    target_layer = model.transformer.h[layer_idx]
    
    # Hook the MLP input projection (c_fc)
    handle_fc = target_layer.mlp.c_fc.register_forward_hook(hook_fn('mlp_input'))
    
    # Forward pass to collect activations
    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    
    # Remove hooks
    handle_fc.remove()
    
    # Get the weight matrices
    fc_weight = target_layer.mlp.c_fc.weight.detach().cpu()  # Shape: [3072, 768] for GPT-2
    proj_weight = target_layer.mlp.c_proj.weight.detach().cpu()  # Shape: [768, 3072]
    
    # Get input activations (reshape to 2D)
    mlp_input = activations['mlp_input'].reshape(-1, activations['mlp_input'].size(-1))  # [batch*seq, hidden]
    
    return fc_weight, proj_weight, mlp_input

# Extract information for layer 0 as example
target_layer_idx = 0
print(f"Extracting layer {target_layer_idx} information...")
fc_weight, proj_weight, mlp_activations = extract_layer_info(model, target_layer_idx, calibration_ids, calibration_mask)

print(f"FC weight shape: {fc_weight.shape}")
print(f"Projection weight shape: {proj_weight.shape}")
print(f"MLP input activations shape: {mlp_activations.shape}")

## 4. Implement Wanda Mask Selection Algorithm

In [ ]:
def wanda_mask_selection(weight_matrix, input_activations, sparsity_ratio):
    """
    Implement Wanda mask selection algorithm
    Args:
        weight_matrix: Weight matrix W of shape [output_dim, input_dim]
        input_activations: Input activations X of shape [num_samples, input_dim]
        sparsity_ratio: Fraction of weights to prune (0.0 to 1.0)
    Returns:
        mask: Binary mask (1 = keep, 0 = prune)
        importance_scores: Weight importance scores
    """
    print(f"Computing Wanda importance scores for sparsity ratio: {sparsity_ratio:.2f}")
    
    # Compute input activation norms (per feature)
    # Shape: [input_dim]
    activation_norms = torch.norm(input_activations, dim=0, p=2)
    
    # Compute weight magnitudes
    # Shape: [output_dim, input_dim]
    weight_magnitudes = torch.abs(weight_matrix)
    
    # Compute importance scores: |W_ij| * ||X_j||_2
    # Broadcasting: [output_dim, input_dim] * [input_dim] -> [output_dim, input_dim]
    importance_scores = weight_magnitudes * activation_norms.unsqueeze(0)
    
    # Flatten importance scores for ranking
    flattened_scores = importance_scores.flatten()
    
    # Determine number of weights to keep
    total_weights = flattened_scores.numel()
    num_keep = int(total_weights * (1 - sparsity_ratio))
    
    # Find threshold for top-k important weights
    topk_values, topk_indices = torch.topk(flattened_scores, num_keep)
    threshold = topk_values[-1].item()
    
    # Create binary mask
    mask = (importance_scores >= threshold).float()
    
    actual_sparsity = 1.0 - mask.mean().item()
    print(f"Target sparsity: {sparsity_ratio:.3f}, Achieved sparsity: {actual_sparsity:.3f}")
    print(f"Weights kept: {int(mask.sum().item()):,} / {total_weights:,}")
    
    return mask, importance_scores

# Test Wanda mask selection on GPT-2 layer
sparsity_ratio = 0.5  # 50% sparsity
wanda_mask, importance_scores = wanda_mask_selection(fc_weight, mlp_activations, sparsity_ratio)

print(f"\nMask statistics:")
print(f"Mask shape: {wanda_mask.shape}")
print(f"Sparsity achieved: {1 - wanda_mask.mean().item():.3f}")

## 5. Implement ADMM Weight Update Algorithm

In [ ]:
def admm_weight_update(original_weights, input_activations, pruning_mask, 
                      max_iterations=20, rho=1.0, tolerance=1e-6, verbose=True):
    """
    ADMM algorithm for optimal weight update under sparsity constraints
    
    Args:
        original_weights: Original weight matrix W [output_dim, input_dim]
        input_activations: Input activations X [num_samples, input_dim]
        pruning_mask: Binary mask M (1=keep, 0=prune)
        max_iterations: Maximum ADMM iterations
        rho: ADMM penalty parameter
        tolerance: Convergence tolerance
        verbose: Print iteration details
    
    Returns:
        optimized_weights: Optimized pruned weights
        convergence_history: Dictionary with convergence metrics
    """
    if verbose:
        print(f"Starting ADMM optimization with rho={rho}, max_iterations={max_iterations}")
    
    # Initialize ADMM variables
    W_tilde = original_weights.clone()  # Primal variable
    Z = original_weights.clone()        # Auxiliary variable
    U = torch.zeros_like(original_weights)  # Dual variable
    
    # Precompute matrices for efficiency
    X = input_activations
    XtX = X.t() @ X  # [input_dim, input_dim]
    XtXW = XtX @ original_weights.t()  # [input_dim, output_dim]
    
    # History tracking
    history = {
        'reconstruction_errors': [],
        'primal_residuals': [],
        'dual_residuals': [],
        'constraint_violations': []
    }
    
    for iteration in range(max_iterations):
        W_old = W_tilde.clone()
        
        # W-update: Minimize reconstruction error + penalty term
        # Solve: (X^T X + rho I) W = X^T X W_orig + rho (Z - U)
        rhs = XtXW + rho * (Z - U).t()  # [input_dim, output_dim]
        lhs_matrix = XtX + rho * torch.eye(XtX.size(0))  # [input_dim, input_dim]
        
        # Solve linear system for each output dimension
        W_tilde_t = torch.linalg.solve(lhs_matrix, rhs)  # [input_dim, output_dim]
        W_tilde = W_tilde_t.t()  # [output_dim, input_dim]
        
        # Z-update: Apply sparsity constraint
        Z = W_tilde + U
        Z = Z * pruning_mask  # Enforce sparsity: Z_ij = 0 where mask_ij = 0
        
        # U-update: Update dual variables
        U = U + (W_tilde - Z)
        
        # Compute convergence metrics
        primal_residual = torch.norm(W_tilde - Z, 'fro').item()
        dual_residual = rho * torch.norm(Z - W_old, 'fro').item()
        
        # Reconstruction error
        recon_error = torch.norm(X @ W_tilde.t() - X @ original_weights.t(), 'fro').item()
        
        # Constraint violation
        constraint_violation = torch.norm(W_tilde * (1 - pruning_mask), 'fro').item()
        
        # Store history
        history['reconstruction_errors'].append(recon_error)
        history['primal_residuals'].append(primal_residual)
        history['dual_residuals'].append(dual_residual)
        history['constraint_violations'].append(constraint_violation)
        
        if verbose and (iteration % 5 == 0 or iteration == max_iterations - 1):
            print(f"Iter {iteration+1:2d}: Recon={recon_error:.6f}, "
                  f"Primal={primal_residual:.6f}, Dual={dual_residual:.6f}")
        
        # Check convergence
        if primal_residual < tolerance and dual_residual < tolerance:
            if verbose:
                print(f"Converged at iteration {iteration+1}")
            break
    
    # Final result: apply mask to ensure exact sparsity
    optimized_weights = W_tilde * pruning_mask
    
    if verbose:
        final_sparsity = 1 - (optimized_weights != 0).float().mean().item()
        print(f"Final sparsity: {final_sparsity:.3f}")
        print(f"ADMM optimization completed in {len(history['reconstruction_errors'])} iterations")
    
    return optimized_weights, history

# Apply ADMM to GPT-2 layer weights
print("Applying ADMM optimization to GPT-2 layer weights...")
optimized_fc_weights, admm_history = admm_weight_update(
    fc_weight, mlp_activations, wanda_mask, 
    max_iterations=20, rho=1.0, verbose=True
)

## 6. Apply Pruning to GPT-2 Model

In [ ]:
def prune_gpt2_layer(model, layer_idx, pruning_mask, optimized_weights):
    """
    Apply pruned weights to a specific GPT-2 layer
    """
    target_layer = model.transformer.h[layer_idx]
    
    # Replace the MLP input projection weights
    with torch.no_grad():
        target_layer.mlp.c_fc.weight.data = optimized_weights.to(device)
    
    print(f"Applied pruned weights to layer {layer_idx}")
    
    # Verify sparsity
    actual_sparsity = (target_layer.mlp.c_fc.weight.data == 0).float().mean().item()
    print(f"Verified sparsity in model: {actual_sparsity:.3f}")
    
    return actual_sparsity

# Create a copy of the model for pruning
pruned_model = copy.deepcopy(model)
sparsity_achieved = prune_gpt2_layer(pruned_model, target_layer_idx, wanda_mask, optimized_fc_weights)

print(f"\nModel pruning completed for layer {target_layer_idx}")
print(f"Sparsity achieved: {sparsity_achieved:.3f}")

## 7. Evaluate Model Performance

In [ ]:
def evaluate_perplexity(model, tokenizer, test_texts, max_length=128):
    """
    Evaluate model perplexity on test texts
    """
    model.eval()
    total_loss = 0
    total_tokens = 0
    
    with torch.no_grad():
        for text in tqdm(test_texts, desc="Evaluating perplexity"):
            if len(text.strip()) < 10:  # Skip very short texts
                continue
                
            inputs = tokenizer(text, max_length=max_length, 
                             truncation=True, return_tensors="pt")
            inputs = {k: v.to(device) for k, v in inputs.items()}
            
            outputs = model(**inputs, labels=inputs['input_ids'])
            loss = outputs.loss
            
            total_loss += loss.item() * inputs['input_ids'].size(1)
            total_tokens += inputs['input_ids'].size(1)
    
    avg_loss = total_loss / total_tokens
    perplexity = torch.exp(torch.tensor(avg_loss)).item()
    
    return perplexity

# Prepare test data
test_dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
test_texts = [example['text'] for example in test_dataset if len(example['text'].strip()) > 50][:20]  # Small subset

print("Evaluating original model...")
original_perplexity = evaluate_perplexity(model, tokenizer, test_texts)

print("Evaluating pruned model...")
pruned_perplexity = evaluate_perplexity(pruned_model, tokenizer, test_texts)

print(f"\nPerplexity Results:")
print(f"Original model: {original_perplexity:.2f}")
print(f"Pruned model:   {pruned_perplexity:.2f}")
print(f"Degradation:    {pruned_perplexity - original_perplexity:.2f} ({((pruned_perplexity/original_perplexity - 1) * 100):.1f}%)")

## 8. Analyze Results and Visualize Performance

In [ ]:
# Plot ADMM convergence
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Reconstruction error
axes[0, 0].plot(admm_history['reconstruction_errors'])
axes[0, 0].set_title('Reconstruction Error')
axes[0, 0].set_xlabel('Iteration')
axes[0, 0].set_ylabel('Error')
axes[0, 0].grid(True)

# Primal residual
axes[0, 1].semilogy(admm_history['primal_residuals'])
axes[0, 1].set_title('Primal Residual')
axes[0, 1].set_xlabel('Iteration')
axes[0, 1].set_ylabel('Log Residual')
axes[0, 1].grid(True)

# Dual residual
axes[1, 0].semilogy(admm_history['dual_residuals'])
axes[1, 0].set_title('Dual Residual')
axes[1, 0].set_xlabel('Iteration')
axes[1, 0].set_ylabel('Log Residual')
axes[1, 0].grid(True)

# Constraint violations
axes[1, 1].semilogy(admm_history['constraint_violations'])
axes[1, 1].set_title('Constraint Violations')
axes[1, 1].set_xlabel('Iteration')
axes[1, 1].set_ylabel('Log Violation')
axes[1, 1].grid(True)

plt.tight_layout()
plt.savefig('admm_convergence.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Compare pruning methods
def compare_pruning_methods(original_weights, input_activations, sparsity_ratio):
    """
    Compare ADMM+Wanda vs simple magnitude pruning
    """
    print("Comparing pruning methods...")
    
    # Method 1: Simple magnitude pruning
    weight_magnitudes = torch.abs(original_weights)
    threshold = torch.quantile(weight_magnitudes, sparsity_ratio)
    magnitude_mask = (weight_magnitudes >= threshold).float()
    magnitude_pruned = original_weights * magnitude_mask
    
    # Method 2: ADMM + Wanda (already computed)
    wanda_pruned = optimized_fc_weights
    
    # Compute reconstruction errors
    X = input_activations
    original_output = X @ original_weights.t()
    
    magnitude_error = torch.norm(X @ magnitude_pruned.t() - original_output, 'fro')**2
    wanda_error = torch.norm(X @ wanda_pruned.t() - original_output, 'fro')**2
    
    print(f"\nPruning Method Comparison (Sparsity: {sparsity_ratio:.1%}):")
    print(f"Magnitude pruning error: {magnitude_error:.6f}")
    print(f"ADMM+Wanda error:       {wanda_error:.6f}")
    print(f"Improvement:             {((magnitude_error - wanda_error) / magnitude_error * 100):.1f}%")
    
    return magnitude_error.item(), wanda_error.item()

mag_error, wanda_error = compare_pruning_methods(fc_weight, mlp_activations, sparsity_ratio)

## 9. Save Results and Pruned Model

In [ ]:
# Save pruning results
results = {
    'model_name': model_name,
    'pruned_layer': target_layer_idx,
    'sparsity_ratio': sparsity_ratio,
    'sparsity_achieved': sparsity_achieved,
    'original_perplexity': original_perplexity,
    'pruned_perplexity': pruned_perplexity,
    'perplexity_degradation': pruned_perplexity - original_perplexity,
    'admm_iterations': len(admm_history['reconstruction_errors']),
    'magnitude_pruning_error': mag_error,
    'wanda_admm_error': wanda_error,
    'reconstruction_improvement': ((mag_error - wanda_error) / mag_error * 100)
}

# Save results to JSON
with open('pruning_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print("Results saved to 'pruning_results.json'")

# Save the pruned model
output_dir = './pruned_gpt2_model'
os.makedirs(output_dir, exist_ok=True)

pruned_model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print(f"Pruned model saved to '{output_dir}'")

# Print summary
print(f"\n{'='*60}")
print(f"PRUNING SUMMARY")
print(f"{'='*60}")
print(f"Model: {model_name}")
print(f"Layer pruned: {target_layer_idx}")
print(f"Sparsity: {sparsity_achieved:.1%}")
print(f"Perplexity change: {original_perplexity:.2f} → {pruned_perplexity:.2f} ({((pruned_perplexity/original_perplexity - 1) * 100):+.1f}%)")
print(f"ADMM vs Magnitude pruning: {results['reconstruction_improvement']:.1f}% improvement")
print(f"ADMM converged in {results['admm_iterations']} iterations")
print(f"{'='*60}")

## 10. Extended Analysis: Multiple Layers

In [ ]:
# Optional: Prune multiple layers
def prune_multiple_layers(model, layer_indices, sparsity_ratio, calibration_ids, calibration_mask):
    """
    Apply ADMM pruning to multiple layers
    """
    pruned_model = copy.deepcopy(model)
    layer_results = {}
    
    for layer_idx in layer_indices:
        print(f"\nProcessing layer {layer_idx}...")
        
        # Extract layer information
        fc_weight, _, mlp_activations = extract_layer_info(pruned_model, layer_idx, calibration_ids, calibration_mask)
        
        # Apply Wanda + ADMM
        mask, _ = wanda_mask_selection(fc_weight, mlp_activations, sparsity_ratio)
        optimized_weights, history = admm_weight_update(fc_weight, mlp_activations, mask, verbose=False)
        
        # Apply to model
        actual_sparsity = prune_gpt2_layer(pruned_model, layer_idx, mask, optimized_weights)
        
        layer_results[layer_idx] = {
            'sparsity': actual_sparsity,
            'iterations': len(history['reconstruction_errors']),
            'final_error': history['reconstruction_errors'][-1]
        }
    
    return pruned_model, layer_results

# Example: Prune first 3 layers (uncomment to run)
# layers_to_prune = [0, 1, 2]
# multi_pruned_model, multi_results = prune_multiple_layers(model, layers_to_prune, 0.5, calibration_ids, calibration_mask)
# print(f"\nMulti-layer pruning results:")
# for layer_idx, results in multi_results.items():
#     print(f"Layer {layer_idx}: {results['sparsity']:.1%} sparsity, {results['iterations']} iterations")

print("\nADMM-based GPT-2 pruning implementation completed successfully!")